# Effect of Microbe Derived Metabolits on Salmonella growth in a Mouse Gut

## Generate Constrained Models of the STM only condition and the Mixed STM condition

In [1]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes
print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu
STM_V1_0 Control Model has 1271 total genes
All models have 1271 genes
All models have 2545 reactions
Read LP format model from file /tmp/tmpumwlnjb7.lp
Reading time = 0.01 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpxdwf0ox1.lp
Reading time = 0.01 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp74akxmtk.lp
Reading time = 0.01 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp813_kul0.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpit4njqsb.lp
Reading time 

In [2]:
# make a dictionary of all the models 

models = {
    "stm_unconstrained": control_model,
    "mixed_stm1090": mixedstm1090_model,
    "mixed_stm1585": mixedstm1585_model,
    "mixed_stm2575": mixedstm2575_model,
    "stm_1090": stm1090_model,
    "stm_1585": stm1585_model,
    "stm_2575": stm2575_model
}

## Define the Mouse Minimal Medium Conditions

In [3]:
# Make a copy of the default medium from stm_v1_0
default_medium = dict(models["stm_unconstrained"].medium)

# Add mouse-derived, gut metabolites to the default medium to create a nouse gut minimal medium

mouse_min_medium = dict(default_medium)
mouse_min_medium.update({
    "EX_o2_e": 0.5, # updating to anoxic conditions
    "EX_no3_e": 0.5,
    "EX_for_e": 4.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_glc__D_e": 0.0 # removing from default medium
})

## Extracting a metabolite utlization summary file for each metabolite in the medium

Declare a variable and a copied model to it. 
Run FBA on the copied model and save the solution to a variable. 
Generate a summary of the FBA solution for the solved model and save to a variable. 

In [4]:
test_model = models["mixed_stm1090"].copy()
mouse_medium = dict(mouse_min_medium)
test_model.medium = mouse_medium
solution = test_model.optimize()
summary = test_model.summary()

Read LP format model from file /tmp/tmp7ipzqi2h.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


Generate a summary dataframe and csv for `atp_c` in `test_model` as an example of the data I want to collect on the medium metabolites. 

In [5]:
metabolite = test_model.metabolites.atp_c.summary(solution=solution)

producing = metabolite.producing_flux.copy()
producing["Type"] = "Producing"

consuming= metabolite.consuming_flux.copy()
consuming["Type"] = "Consuming"

metabolite_df = pd.concat(
    [producing, consuming],
    ignore_index=True
)   

metabolite_df["Reaction Name"] = pd.NA
metabolite_df["Reaction Name"] = metabolite_df["reaction"].map(
    lambda rxn_id: test_model.reactions.get_by_id(str(rxn_id)).name
)
metabolite_df["Objective Value"] = pd.NA
metabolite_df.loc[0, "Objective Value"] = solution.objective_value

metabolite_df.to_csv(
    f"~/ATP_C_minimal_media_summary_flux.csv",
        index=False
    )

## Using the solved model medium to go from reaction objects in the medium to metabolite objects to call the metabolite.summary() method

In `model.medium` the medium is a dictionary with reaction_ids (strings) for they keys and the reaction bound value (float) for the value. I need to extract the reaction_id string and call `model.reactions.get_by_id(reaction_id)` to get the reaction object. 

In [6]:
# copy the medium for the solved model
medium = mouse_min_medium.copy()

# make a list of the keys in the medium 
rxn_ids = list(medium.keys())
print(rxn_ids)
print(type(rxn_ids[0]))

['EX_ca2_e', 'EX_cbl1_e', 'EX_cl_e', 'EX_co2_e', 'EX_cobalt2_e', 'EX_cu2_e', 'EX_mg2_e', 'EX_fe2_e', 'EX_fe3_e', 'EX_mn2_e', 'EX_mobd_e', 'EX_na1_e', 'EX_nh4_e', 'EX_o2_e', 'EX_pi_e', 'EX_zn2_e', 'EX_so4_e', 'EX_glc__D_e', 'EX_tungs_e', 'EX_h2o_e', 'EX_h_e', 'EX_k_e', 'EX_no3_e', 'EX_for_e', 'EX_asp__L_e', 'EX_mal__L_e', 'EX_galct__D_e', 'EX_glcr_e', 'EX_pyr_e', 'EX_arab__L_e']
<class 'str'>


In [7]:
# Use reaction_ids (list of strings) to get the reaction objects in a new list.  

rxn_obj_list = []

for rxn in rxn_ids:
    rxn_obj = test_model.reactions.get_by_id(rxn)
    rxn_obj_list.append(rxn_obj)

print(rxn_obj_list)

[<Reaction EX_ca2_e at 0xf2e6ca6c610>, <Reaction EX_cbl1_e at 0xf2e6ca6c950>, <Reaction EX_cl_e at 0xf2e6ca6e8d0>, <Reaction EX_co2_e at 0xf2e6ca6ebd0>, <Reaction EX_cobalt2_e at 0xf2e6ca6eed0>, <Reaction EX_cu2_e at 0xf2e6ca841d0>, <Reaction EX_mg2_e at 0xf2e6ca92a10>, <Reaction EX_fe2_e at 0xf2e6ca93490>, <Reaction EX_fe3_e at 0xf2e6ca93610>, <Reaction EX_mn2_e at 0xf2e6ca93a90>, <Reaction EX_mobd_e at 0xf2e6ca93f10>, <Reaction EX_na1_e at 0xf2e6caa0b50>, <Reaction EX_nh4_e at 0xf2e6caa15d0>, <Reaction EX_o2_e at 0xf2e6caa2650>, <Reaction EX_pi_e at 0xf2e6caacc10>, <Reaction EX_zn2_e at 0xf2e6caadc90>, <Reaction EX_so4_e at 0xf2e6cabda10>, <Reaction EX_glc__D_e at 0xf2e6ca17ed0>, <Reaction EX_tungs_e at 0xf2e6caced10>, <Reaction EX_h2o_e at 0xf2e6cf416d0>, <Reaction EX_h_e at 0xf2e6cf53810>, <Reaction EX_k_e at 0xf2e6fc84650>, <Reaction EX_no3_e at 0xf2e6caa21d0>, <Reaction EX_for_e at 0xf2e6caa18d0>, <Reaction EX_asp__L_e at 0xf2e6ca63990>, <Reaction EX_mal__L_e at 0xf2e6fc9e8d0>, <

Confirm the items in `rxn_obj_list` are reaction objects.

In [8]:
ca2 = (rxn_obj_list[0])
print(ca2)
print(type(ca2))

EX_ca2_e: ca2_e <=> 
<class 'cobra.core.reaction.Reaction'>


Figure out how to go from a reaction object to metabolite object using the reaction object example of `ca2`

In [9]:
# copy the dictionary of metabolites for a reaction object. The key is a metabolite object and the value is a metabolite uptake value (float).
ca2_metabolite_dict = dict(ca2.metabolites)
# print the dictionary
print("The type for `reaction.metabolites` is", type(ca2_metabolite_dict))
#print the datatype for `ca2_metabolite_dict`
print("The dictionary of metabolites for reaction `ca2` is", ca2_metabolite_dict)

# make a list of the keys in `ca2_metabolite_dict` (Method 1)
# This method uses the reaction.metabolite dictionary to make a list of metabolite keys. 
ca2_keys_list1 = list(ca2_metabolite_dict.keys())
# check the list has the same key from `ca2_metabolite_dict`
print("The copied list of key from `ca2.metabolites.keys()` is", ca2_keys_list1)
# check the datatype for the first item at the first position ([0] index)
print("The type for the first item of`list(ca2.metabolites.keys())` is", type(ca2_keys_list1[0]))

# Question: What's the difference between specifying index [0] and not doing it? (Method 2)
# This method uses the reaction.metabolite dictionary to make a list of metabolite keys. 
ca2_keys_list2 = list(ca2_metabolite_dict.keys())[0]
print("The first item of list`ca2_metabolite1` is type", type(ca2_keys_list2))
print("The metabolite for list `ca2_metabolite1' is", ca2_keys_list2)

# What's the difference between methods 1,2, and 3? (Method 3)
# This method gets the metabolite object from the dictionary of metabolites in a reaction and saves it as a variable. 
ca2_metabolite3 = next(iter(ca2_metabolite_dict.keys()))
print("The type for `next(iter(dict(reaction.metabolites)))` is", type(ca2_metabolite3))
print("The metabolite is", ca2_metabolite3)

# Can I check the type for the key in `ca2_metabolite_dict` without making a list?
types = [type(k) for k in ca2_metabolite_dict.keys()]
print("The type for `types` is", type(types))
print("The items in list `types` are", types)


The type for `reaction.metabolites` is <class 'dict'>
The dictionary of metabolites for reaction `ca2` is {<Metabolite ca2_e at 0xf2e6cf33410>: -1.0}
The copied list of key from `ca2.metabolites.keys()` is [<Metabolite ca2_e at 0xf2e6cf33410>]
The type for the first item of`list(ca2.metabolites.keys())` is <class 'cobra.core.metabolite.Metabolite'>
The first item of list`ca2_metabolite1` is type <class 'cobra.core.metabolite.Metabolite'>
The metabolite for list `ca2_metabolite1' is ca2_e
The type for `next(iter(dict(reaction.metabolites)))` is <class 'cobra.core.metabolite.Metabolite'>
The metabolite is ca2_e
The type for `types` is <class 'list'>
The items in list `types` are [<class 'cobra.core.metabolite.Metabolite'>]


I can use the reaction object `ca2` from `rxn_obj_list` to get a dictionary of metabolite object keys and flux float values for the `ca2` reaction object. 

Below is an example of the full process. The reaction id string is used to call `model.reactions.get_by_id()`, the reaction object is used to call `reaction.metabolites` which creates a dictionary. The keys are metabolite objects which are copied into a list. The list of metabolite objects is used to call `metabolite.summary()` for each. 

In [10]:
ca2_rxn = test_model.reactions.get_by_id(rxn_ids[0])
ca2_dict = ca2_rxn.metabolites
ca2_metabolite = list(ca2.metabolites.keys())[0]
ca2_sum = ca2_metabolite.summary()
print(ca2_sum)
print(ca2_metabolite)
print(type(ca2_rxn))
print(type(ca2_dict))
print(type(ca2_metabolite))
print(type(ca2_sum))



ca2_e
=====
Formula: Ca

Producing Reactions
-------------------
Percent     Flux Reaction Definition
100.00% 0.003651 EX_ca2_e ca2_e <=> 

Consuming Reactions
-------------------
Percent      Flux Reaction      Definition
100.00% -0.003651   CA2tex ca2_e <=> ca2_p
ca2_e
<class 'cobra.core.reaction.Reaction'>
<class 'dict'>
<class 'cobra.core.metabolite.Metabolite'>
<class 'cobra.summary.metabolite_summary.MetaboliteSummary'>


In [11]:
print(ca2_dict)

{<Metabolite ca2_e at 0xf2e6cf33410>: -1.0}


In [12]:
sum_df = ca2_sum.producing_flux.copy()
sum_dict = sum_df.to_dict
print(sum_dict)


<bound method DataFrame.to_dict of               flux  reaction  percent
EX_ca2_e  0.003651  EX_ca2_e      1.0>


## A for loop to generate metabolite summaries from a list of reaction ids - work in progress 09.19.26

The idea below is to start with a list of reaction_ids pulled from the solved models' medium, iterate through that list, and create a new list of reaction objects. The reaction objects list is used to iterate through the reaction objects to make a dictionary of metabolites in each reaction. The keys in the the metabolites dictionary are metabolite objects. The keys are iterated over to call the `.summary()` method on the metabolite object. The result is a dictionary of metabolite summaries as values and the reaction object as the key. 

Where I'm struggling now is figuring out how to "save" these metabolite summaries. Below is one idea where I iterate through the `metabolite_summaries_dict`, pull the producing flux and consuming flux and save a copy of each dataframe and then convert each dataframe to a dictionary. I then append the producing_flux dicitonary and consuming_flux dictionary to `producing_consuming_list`. A list for each reaction is added to a dictionary, `all_metabolites_summary`. 

What I don't like about this approach is that each reaction object has a list and each list has two dictionaries. It just feels confusing to then figure out how to put this into csv. 

I also have some concerns on what this is actually doing: 
`producing_dict` and `consuming_dict` are variables that get reused for every reaction object in `metabolite_summaries_dict`. The dictionaries are appended to `producing_consuming_list` each time it iterates over a reaction object. The list then becomes a value for each reaction object key in the `all_metabolites_summary` dictionary. I'm wondering if 

What I'd rather do is 

In [13]:
import pandas as pd

rxn_obj_list = []
for rxn in rxn_ids:
    rxn_obj = test_model.reactions.get_by_id(rxn)
    rxn_obj_list.append(rxn_obj)
    

metabolite_summaries_dict = {}
for rxn_obj in rxn_obj_list:
    metabolite_dict = dict(rxn_obj.metabolites)
    metabolite_obj = next(iter(metabolite_dict.keys()))
    metabolite_summary = metabolite_obj.summary(solution = solution)
    metabolite_summaries_dict[rxn_obj] = metabolite_summary


# the general metabolite summary is not a dictionary that can me saved. 

producing_consuming_list = []
all_metabolites_summary = {}
for rxn_obj, metabolite_summary in metabolite_summaries_dict.items():
    producing_df = metabolite_summary.producing_flux.copy()
    producing_dict = producing_df.to_dict()
    producing_consuming_list.append(producing_dict.copy())

    consuming_df = metabolite_summary.consuming_flux.copy()
    consuming_dict = consuming_df.to_dict()
    producing_consuming_list.append(consuming_dict.copy())

    all_metabolites_summary[rxn_obj] = producing_consuming_list

print(producing_consuming_list)

[{'flux': {'EX_ca2_e': 0.0036507312948327285}, 'reaction': {'EX_ca2_e': 'EX_ca2_e'}, 'percent': {'EX_ca2_e': 1.0}}, {'flux': {'CA2tex': -0.0036507312948327285}, 'reaction': {'CA2tex': 'CA2tex'}, 'percent': {'CA2tex': 1.0}}, {'flux': {}, 'reaction': {}, 'percent': {}}, {'flux': {'CBL1tonex': 0.0, 'EX_cbl1_e': 0.0}, 'reaction': {'CBL1tonex': 'CBL1tonex', 'EX_cbl1_e': 'EX_cbl1_e'}, 'percent': {'CBL1tonex': nan, 'EX_cbl1_e': nan}}, {'flux': {'EX_cl_e': 0.0036507312948203742}, 'reaction': {'EX_cl_e': 'EX_cl_e'}, 'percent': {'EX_cl_e': 1.0}}, {'flux': {'CLtex': -0.0036507312948203742}, 'reaction': {'CLtex': 'CLtex'}, 'percent': {'CLtex': 1.0}}, {'flux': {'CO2tex': 9.73578491477163}, 'reaction': {'CO2tex': 'CO2tex'}, 'percent': {'CO2tex': 1.0}}, {'flux': {'EX_co2_e': -9.73578491477163}, 'reaction': {'EX_co2_e': 'EX_co2_e'}, 'percent': {'EX_co2_e': 1.0}}, {'flux': {'EX_cobalt2_e': 0.002433820863213583}, 'reaction': {'EX_cobalt2_e': 'EX_cobalt2_e'}, 'percent': {'EX_cobalt2_e': 1.0}}, {'flux': {

In [14]:
ca2_producing = producing_consuming_list[0].copy()

`all_metabolites_summary` is a dictionary where the keys are reaction objects and the value is a list dictionaries. The list is a producing dictionary and the other is a consuming dictionary. 

## Adding metabolite utilization extraction for loop to full results extraction loop

In [15]:
metabolite_summaries = {}
#counter = 0

for rxn_id in reactions:
    rxn = test_model.reactions.get_by_id(rxn_id)
    rxn_metabolite = rxn.metabolites
    metabolite = list(rxn_metabolite.keys())[0]

    metabolite_summary = metabolite.summary(solution=solution)

    producing = metabolite_summary.producing_flux.copy()
    producing["Type"] = "Producing"

    consuming= metabolite_summary.consuming_flux.copy()
    consuming["Type"] = "Consuming"

    metabolite_summary_df = pd.concat(
    [producing, consuming],
    ignore_index=True
    )

    metabolite_summary_df["Reaction Name"] = pd.NA
    metabolite_summary_df["Reaction Name"] = metabolite_summary_df["reaction"].map(
    lambda rxn_id: test_model.reactions.get_by_id(str(rxn_id)).name
    )
    metabolite_summary_df["Objective Value"] = pd.NA
    metabolite_summary_df.loc[0, "Objective Value"] = solution.objective_value

    metabolite_summaries[rxn_id] = metabolite_summary_df

    #print(metabolite_summary_df)
    metabolite_summary_df.to_csv(
        f"/home/gmvaz/2026_GEMs/stm_model_test/test_metabolite_flux.csv",
            index=False
        )

print(metabolite_summaries)

NameError: name 'reactions' is not defined

In [ ]:
print(type(metabolite_summaries))

<class 'dict'>


In [ ]:
metabolite_summaries = {}

for model_name, model in models.items():
    test_model = model.copy()
    mouse_medium = dict(mouse_min_medium)
    test_model.medium = mouse_medium
    solution = test_model.optimize()
    rxn_ids = list(mouse_medium.keys())

    for rxn_id in rxn_ids:
        rxn = test_model.reactions.get_by_id(rxn_id)

        rxn_metabolite = rxn.metabolites

        assert len(rxn_metabolite) == 1

        metabolite = list(rxn_metabolite.keys())[0]

        metabolite_summary = metabolite.summary(solution=solution)

        # Producing fluxes
        producing = metabolite_summary.producing_flux.copy()
        producing["Type"] = "Producing"

        # Consuming fluxes
        consuming = metabolite_summary.consuming_flux.copy()
        consuming["Type"] = "Consuming"

        # Combine producing and consuming fluxes
        metabolite_summary_df = pd.concat(
            [producing, consuming],
            ignore_index=True
        )

        # Add metabolite information
        metabolite_summary_df["Metabolite ID"] = metabolite.id
        metabolite_summary_df["Metabolite Name"] = metabolite.name

        # Add reaction names
        metabolite_summary_df["Reaction Name"] = (
            metabolite_summary_df["reaction"].map(
                lambda rxn_id: test_model.reactions.get_by_id(str(rxn_id)).name
            )
        )

        # Add objective value
        metabolite_summary_df["Objective Value"] = solution.objective_value
        metabolite_summary_df["Model Name"] = model_name

        # Store each summary in the dictionary
        metabolite_summaries[(rxn_id, model_name)] = metabolite_summary_df
    


# Combine ALL metabolite summaries into one DataFrame
all_metabolite_summaries_df = pd.concat(
    metabolite_summaries.values(),
    ignore_index=True
)

# Save all summaries in one CSV
all_metabolite_summaries_df.to_csv(
    "/home/gmvaz/2026_GEMs/stm_model_test/test_metabolite_flux.csv",
    index=False
)

# Display combined DataFrame
all_metabolite_summaries_df

Read LP format model from file /tmp/tmpwukxqncm.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


,flux,reaction,percent,Type,Metabolite ID,Metabolite Name,Reaction Name,Objective Value,Model Name
0,0.003832,EX_ca2_e,1.0,Producing,ca2_e,Calcium,Calcium exchange,0.809029,stm_unconstrained
1,-0.003832,CA2tex,1.0,Consuming,ca2_e,Calcium,Calcium (Ca+2) transport via diffusion (extrac...,0.809029,stm_unconstrained
2,0.000000,CBL1tonex,NaN,Consuming,cbl1_e,Cob(I)alamin,Cob(1)alamin transport via ton system (extermal),0.809029,stm_unconstrained
3,0.000000,EX_cbl1_e,NaN,Consuming,cbl1_e,Cob(I)alamin,Cob(I)alamin exchange,0.809029,stm_unconstrained
4,0.003832,EX_cl_e,1.0,Producing,cl_e,Chloride,Chloride exchange,0.809029,stm_unconstrained
...,...,...,...,...,...,...,...,...,...
71,-10.000000,GLCRtex,1.0,Consuming,glcr_e,D-Glucarate,D-glucarate transport via diffusion (extracell...,0.809029,stm_unconstrained
72,10.000000,EX_pyr_e,1.0,Producing,pyr_e,Pyruvate,Pyruvate exchange,0.809029,stm_unconstrained
73,-10.000000,PYRtex,1.0,Consuming,pyr_e,Pyruvate,Pyruvate transport via diffusion (extracellula...,0.809029,stm_unconstrained
74,13.300000,EX_arab__L_e,1.0,Producing,arab__L_e,L-Arabinose,L-Arabinose exchange,0.809029,stm_unconstrained


In [20]:
all_metabolite_summaries_df.style

,flux,reaction,percent,Type,Metabolite ID,Metabolite Name,Reaction Name,Objective Value,Model Name
0,0.003832,EX_ca2_e,1.000000,Producing,ca2_e,Calcium,Calcium exchange,0.809029,stm_unconstrained
1,-0.003832,CA2tex,1.000000,Consuming,ca2_e,Calcium,Calcium (Ca+2) transport via diffusion (extracellular to periplasm),0.809029,stm_unconstrained
2,0.000000,CBL1tonex,nan,Consuming,cbl1_e,Cob(I)alamin,Cob(1)alamin transport via ton system (extermal),0.809029,stm_unconstrained
3,0.000000,EX_cbl1_e,nan,Consuming,cbl1_e,Cob(I)alamin,Cob(I)alamin exchange,0.809029,stm_unconstrained
4,0.003832,EX_cl_e,1.000000,Producing,cl_e,Chloride,Chloride exchange,0.809029,stm_unconstrained
5,-0.003832,CLtex,1.000000,Consuming,cl_e,Chloride,Chloride (Cl-1) transport via diffusion (extracellular to periplasm),0.809029,stm_unconstrained
6,8.205151,CO2tex,1.000000,Producing,co2_e,CO2 CO2,CO2 transport via diffusion (extracellular to periplasm),0.809029,stm_unconstrained
7,-8.205151,EX_co2_e,1.000000,Consuming,co2_e,CO2 CO2,CO2 exchange,0.809029,stm_unconstrained
8,0.002555,EX_cobalt2_e,1.000000,Producing,cobalt2_e,Co2+,Co2+ exchange,0.809029,stm_unconstrained
9,-0.002555,COBALT2tex,1.000000,Consuming,cobalt2_e,Co2+,Cobalt (Co+2) transport via diffusion (extracellular to periplasm),0.809029,stm_unconstrained
